# <font color="#418FDE" size="6.5" uppercase>**KNN and Trees**</font>

>Last update: 20260708.
    
By the end of this Lecture, you will be able to:
- Implement k-nearest neighbors classification using manual distance calculations. 
- Explain how simple decision tree splits create interpretable engineering rules. 
- Compare logistic regression, k-nearest neighbors, and simple tree rules using test metrics. 


## **1. KNN From Scratch**

### **1.1. Distance Calculations**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_01_01.jpg?v=1783561635" width="250">



>* KNN compares new cases to known examples
>* Closest neighbors guide the classification decision

>* Features need comparable scales for distance
>* Unscaled large values can dominate similarity

>* Compare unknown cases to all known cases
>* Similarity depends on features, scaling, and distance



In [ ]:
#@title Python Code - Distance Calculations

# Manual distances reveal nearest engineering examples.
# Scaling prevents large units dominating similarity.
# This example uses tiny bridge inspection data.

import math
import pandas as pd

# Create labeled bridge cases with mixed units.
training_rows = [
    {"id": "A", "crack_mm": 1.0, "age_years": 5, "traffic_k": 12, "risk": "low"},
    {"id": "B", "crack_mm": 2.5, "age_years": 12, "traffic_k": 25, "risk": "low"},

    {"id": "C", "crack_mm": 6.0, "age_years": 30, "traffic_k": 55, "risk": "high"},
    {"id": "D", "crack_mm": 8.0, "age_years": 40, "traffic_k": 70, "risk": "high"},
]

# Store the unknown bridge inspection reading.
new_case = {"crack_mm": 5.0, "age_years": 28, "traffic_k": 50}
features = ["crack_mm", "age_years", "traffic_k"]

# Convert records into a small dataframe.
df = pd.DataFrame(training_rows)
assert len(df) == 4 and len(features) == 3

# Compute feature means and standard deviations.
means = {name: df[name].mean() for name in features}
stdevs = {name: df[name].std(ddof=0) for name in features}
assert all(stdevs[name] > 0 for name in features)

# Define a clear Euclidean distance function.
def scaled_distance(row, case, feature_names):
    squared_parts = []
    for name in feature_names:
        row_scaled = (row[name] - means[name]) / stdevs[name]

        case_scaled = (case[name] - means[name]) / stdevs[name]
        squared_parts.append((row_scaled - case_scaled) ** 2)
    return math.sqrt(sum(squared_parts))

# Calculate distance from the new bridge.
df["distance"] = df.apply(
    lambda row: scaled_distance(row, new_case, features), axis=1
)

# Sort cases from closest to farthest.
nearest = df.sort_values("distance").reset_index(drop=True)
top_three = nearest.head(3)

# Vote using the three nearest neighbors.
predicted_risk = top_three["risk"].mode().iloc[0]
summary = top_three[["id", "risk", "distance"]].copy()

# Print compact teaching results only.
print("Scaled distances for the new bridge case:")
for _, row in summary.iterrows():
    print(f"Bridge {row['id']}: {row['risk']}, distance {row['distance']:.2f}")

print(f"KNN prediction with k=3: {predicted_risk} risk")



### **1.2. Distance Calculation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_01_02.jpg?v=1783561637" width="250">



>* KNN classifies by comparing feature distances
>* Nearby labeled examples guide the prediction

>* Feature scales can distort distance comparisons
>* Scaling choices shape meaningful similarity

>* Compare unknown cases with labeled examples
>* Check features, scaling, and distance choice



In [ ]:
#@title Python Code - Distance Calculation

# Manual distances reveal nearest engineering examples.
# Feature scaling prevents misleading distance dominance.
# This script classifies one bridge inspection.

import math

# Store labeled bridge inspection examples.
examples = [
    {"name": "Bridge A", "crack_mm": 1.0, "deflection_mm": 4.0, "label": "safe"},
    {"name": "Bridge B", "crack_mm": 2.0, "deflection_mm": 6.0, "label": "safe"},
    {"name": "Bridge C", "crack_mm": 7.0, "deflection_mm": 18.0, "label": "repair"},
    {"name": "Bridge D", "crack_mm": 8.0, "deflection_mm": 20.0, "label": "repair"},
]

# Define the new inspection case.
new_case = {"crack_mm": 3.0, "deflection_mm": 8.0}

# Validate the tiny teaching dataset.
required_keys = {"crack_mm", "deflection_mm", "label"}
assert all(required_keys <= set(row) for row in examples)
assert len(examples) >= 3

# Calculate Euclidean distance manually.
def euclidean_distance(row, target):
    crack_gap = row["crack_mm"] - target["crack_mm"]
    deflection_gap = row["deflection_mm"] - target["deflection_mm"]
    squared_sum = crack_gap ** 2 + deflection_gap ** 2
    return math.sqrt(squared_sum)

# Attach distances to each labeled example.
distances = []
for row in examples:
    distance = euclidean_distance(row, new_case)
    distances.append((distance, row["name"], row["label"]))

# Sort examples from nearest to farthest.
distances.sort(key=lambda item: item[0])

# Use the three nearest neighbors.
k_value = 3
neighbors = distances[:k_value]
labels = [item[2] for item in neighbors]
prediction = max(set(labels), key=labels.count)

# Print compact teaching results.
print("Manual KNN distance calculation")
print(f"New case: {new_case}")
print("Nearest neighbors:")
for distance, name, label in neighbors:
    print(f"{name}: distance={distance:.2f}, label={label}")
print(f"Predicted class with k={k_value}: {prediction}")



### **1.3. Voting Neighbors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_01_03.jpg?v=1783561640" width="250">



>* Nearest examples guide the predicted class
>* Most common neighbor label becomes the decision

>* Few neighbors capture detail but risk noise
>* More neighbors smooth predictions but may overgeneralize

>* Neighbors may vote equally or by closeness
>* Resolve ties with a consistent rule



In [ ]:
#@title Python Code - Voting Neighbors

# This example demonstrates neighbor voting.
# Distances become labels through voting.
# Small civil data keeps learning clear.

import math
from collections import Counter

# Store training cases as compact dictionaries.
training_cases = [
    {"id": "A", "crack_mm": 1.0, "deflection_mm": 2.0, "label": "safe"},
    {"id": "B", "crack_mm": 1.5, "deflection_mm": 2.5, "label": "safe"},

    {"id": "C", "crack_mm": 4.0, "deflection_mm": 6.0, "label": "repair"},
    {"id": "D", "crack_mm": 4.5, "deflection_mm": 5.5, "label": "repair"},
]

# Define the new inspection measurement.
new_case = {"crack_mm": 3.8, "deflection_mm": 5.7}

# Confirm enough examples exist for voting.
k_neighbors = 3
assert len(training_cases) >= k_neighbors

# Compute Euclidean distances manually.
distances = []
for case in training_cases:
    crack_gap = case["crack_mm"] - new_case["crack_mm"]
    deflection_gap = case["deflection_mm"] - new_case["deflection_mm"]

    squared_sum = crack_gap ** 2 + deflection_gap ** 2
    distance = math.sqrt(squared_sum)
    distances.append((distance, case["id"], case["label"]))

# Sort cases from nearest to farthest.
distances.sort(key=lambda item: item[0])
nearest_neighbors = distances[:k_neighbors]

# Count labels among selected neighbors.
votes = Counter(label for _, _, label in nearest_neighbors)
winner_label, winner_votes = votes.most_common(1)[0]

# Show only concise teaching output.
print("New inspection: crack=3.8 mm, deflection=5.7 mm")
print("Nearest neighbors used for voting:")

# Print each selected neighbor compactly.
for distance, case_id, label in nearest_neighbors:
    print(f"{case_id}: distance={distance:.2f}, label={label}")

# Summarize the vote and prediction.
print(f"Vote counts: {dict(votes)}")
print(f"Predicted class: {winner_label} ({winner_votes} of {k_neighbors} votes)")



## **2. Simple Tree Rules**

### **2.1. Tree Split Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_02_01.jpg?v=1783561627" width="250">



>* Split data using one feature question
>* Branches show the classification reasoning

>* Good splits create clearer class groups
>* Trees choose questions that reduce confusion

>* Tree splits create inspectable engineering rules
>* Validate simple rules with test data



In [ ]:
#@title Python Code - Tree Split Basics

# Simple tree splits create readable engineering rules.
# Tiny sensor data keeps calculations transparent.
# One plot shows the chosen threshold.

import numpy as np
import matplotlib.pyplot as plt

# Store vibration readings and binary labels.
vibration = np.array([0.8, 1.0, 1.2, 1.5, 1.7, 2.1, 2.4, 2.8])
labels = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Confirm matching observation counts before splitting.
assert vibration.shape[0] == labels.shape[0]
assert vibration.size >= 4

# Define candidate thresholds between sorted readings.
sorted_values = np.sort(vibration)
thresholds = (sorted_values[:-1] + sorted_values[1:]) / 2

# Measure impurity using the Gini formula.
def gini(group_labels):
    if len(group_labels) == 0:
        return 0.0

    p_concern = np.mean(group_labels == 1)
    p_normal = 1.0 - p_concern
    return 1.0 - p_normal**2 - p_concern**2

# Score each possible split by weighted impurity.
best_threshold = None
best_score = float("inf")

for threshold in thresholds:
    left_labels = labels[vibration <= threshold]
    right_labels = labels[vibration > threshold]

    left_weight = len(left_labels) / len(labels)
    right_weight = len(right_labels) / len(labels)
    score = left_weight * gini(left_labels) + right_weight * gini(right_labels)

    if score < best_score:
        best_score = score
        best_threshold = threshold

# Convert the best split into an engineering rule.
predicted = (vibration > best_threshold).astype(int)
accuracy = np.mean(predicted == labels)

# Print a compact interpretation for learners.
print(f"Best split: vibration > {best_threshold:.2f} cm/s")
print(f"Weighted Gini after split: {best_score:.3f}")
print(f"Training accuracy for this one-rule tree: {accuracy:.2f}")
print("Rule: higher vibration suggests maintenance concern.")

# Plot observations and the selected threshold.
colors = np.where(labels == 1, "tomato", "steelblue")
plt.figure(figsize=(7, 3))
plt.scatter(vibration, labels, c=colors, s=90)

# Add a vertical line for the split.
plt.axvline(best_threshold, color="black", linestyle="--")
plt.yticks([0, 1], ["normal", "concern"])
plt.xlabel("Vibration level, centimeters per second")

# Finish the plot with a clear title.
plt.title("A simple decision tree split becomes a rule")
plt.tight_layout()
plt.show()



### **2.2. Interpretable Split Rules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_02_02.jpg?v=1783561623" width="250">



>* Splits ask simple feature-based questions
>* Tree paths make model reasoning inspectable

>* Splits use real measurable conditions
>* Readable rules support practical risk decisions

>* Transparent rules support human judgment
>* Check rules for accuracy, fairness, reliability



In [ ]:
#@title Python Code - Interpretable Split Rules

# Simple tree rules support engineering decisions.
# Each split becomes readable threshold logic.
# Tiny data keeps the example transparent.

import numpy as np
import matplotlib.pyplot as plt

# Store bridge observations as compact arrays.
crack_mm = np.array([1, 2, 3, 4, 6, 7, 9, 10])
deflection_in = np.array([0.05, 0.08, 0.10, 0.18, 0.22, 0.35, 0.40, 0.55])
needs_repair = np.array([0, 0, 0, 0, 1, 1, 1, 1])

# Validate matching observation counts before splitting.
assert crack_mm.size == deflection_in.size == needs_repair.size
assert crack_mm.size <= 20

# Define one interpretable engineering split rule.
crack_limit_mm = 5
predicted_repair = (crack_mm > crack_limit_mm).astype(int)

# Compute simple classification metrics manually.
correct_count = int(np.sum(predicted_repair == needs_repair))
accuracy = correct_count / needs_repair.size
false_alarms = int(np.sum((predicted_repair == 1) & (needs_repair == 0)))
missed_repairs = int(np.sum((predicted_repair == 0) & (needs_repair == 1)))

# Print concise rule interpretation and metrics.
print(f"Rule: if crack width > {crack_limit_mm} mm, predict repair.")
print(f"Accuracy on tiny test set: {accuracy:.2f}")
print(f"False alarms: {false_alarms}; missed repairs: {missed_repairs}")
print("This split is readable, inspectable, and challengeable.")

# Plot observations and the decision threshold.
colors = np.where(needs_repair == 1, "tomato", "steelblue")
plt.figure(figsize=(6, 4))
plt.scatter(crack_mm, deflection_in, c=colors, s=80)
plt.axvline(crack_limit_mm, color="black", linestyle="--")
plt.xlabel("Crack width (mm)")
plt.ylabel("Deflection (inches)")
plt.title("Interpretable split rule for bridge repair")
plt.grid(True, alpha=0.3)
plt.show()



### **2.3. Readable Engineering Rules**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_02_03.jpg?v=1783561625" width="250">



>* Tree splits create clear condition-based rules
>* Rules help engineers explain and inspect predictions

>* Tree paths become clear prediction rules
>* Familiar variables connect conditions to outcomes

>* Readable rules support validation and fairness checks
>* Clear tradeoffs guide responsible engineering decisions



In [ ]:
#@title Python Code - Readable Engineering Rules

# Simple tree rules support engineering decisions.
# Thresholds become readable field conditions.
# Small examples keep interpretation clear.

import pandas as pd
import matplotlib.pyplot as plt

# Create a tiny pavement inspection dataset.
data = pd.DataFrame({
    "traffic_kvpd": [8, 12, 18, 22, 30, 35, 40, 45],
    "condition_score": [88, 76, 70, 62, 55, 48, 42, 35],
    "urgent_review": [0, 0, 0, 1, 1, 1, 1, 1],
})

# Validate the expected teaching columns.
needed = {"traffic_kvpd", "condition_score", "urgent_review"}
assert needed.issubset(data.columns), "Required columns are missing."

# Define one simple interpretable tree rule.
def tree_rule(row):
    high_traffic = row["traffic_kvpd"] >= 20
    poor_condition = row["condition_score"] < 65
    return int(high_traffic and poor_condition)

# Apply the readable rule to every segment.
data["tree_prediction"] = data.apply(tree_rule, axis=1)
correct = (data["tree_prediction"] == data["urgent_review"]).sum()
accuracy = correct / len(data)

# Translate the split path into plain language.
rule_text = "IF traffic >= 20 kvpd AND condition < 65 THEN urgent review"
print("Readable tree rule:")
print(rule_text)
print(f"Accuracy on tiny example: {accuracy:.2f}")
print("Example segment 4 prediction:", data.loc[3, "tree_prediction"])

# Plot the two thresholds as engineering boundaries.
colors = data["urgent_review"].map({0: "steelblue", 1: "tomato"})
plt.figure(figsize=(6, 4))
plt.scatter(data["traffic_kvpd"], data["condition_score"], c=colors, s=80)
plt.axvline(20, color="black", linestyle="--", label="traffic split")
plt.axhline(65, color="gray", linestyle="--", label="condition split")
plt.xlabel("Traffic volume, thousand vehicles per day")
plt.ylabel("Pavement condition score")
plt.title("Readable Simple Tree Rule")
plt.legend()
plt.show()



## **3. Classifier Comparison**

### **3.1. Model Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_03_01.jpg?v=1783561629" width="250">



>* Accuracy starts classifier comparison
>* Error types matter in context

>* Precision and recall show different tradeoffs
>* Models balance performance and interpretability differently

>* Use confusion matrices to see error patterns
>* Choose models based on real error costs



In [ ]:
#@title Python Code - Model Metrics

# Compare classifiers using simple test metrics.
# Civil examples often need balanced decisions.
# This script avoids machine learning libraries.

import numpy as np
import pandas as pd

# Create a tiny engineering inspection dataset.
np.random.seed(7)
n_samples = 60

# Simulate bridge inspection measurements.
crack_mm = np.random.normal(2.8, 1.1, n_samples)
load_tons = np.random.normal(18.0, 4.0, n_samples)

# Define risk using an interpretable rule.
risk_score = crack_mm + 0.12 * load_tons
labels = (risk_score > 5.0).astype(int)

# Combine features into one matrix.
features = np.column_stack([crack_mm, load_tons])
assert features.shape[0] == labels.size

# Split data deterministically into train and test.
indices = np.arange(n_samples)
np.random.shuffle(indices)

train_idx = indices[:42]
test_idx = indices[42:]

X_train = features[train_idx]
y_train = labels[train_idx]

X_test = features[test_idx]
y_test = labels[test_idx]

# Standardize features using training statistics only.
means = X_train.mean(axis=0)
stds = X_train.std(axis=0) + 1e-9

X_train_s = (X_train - means) / stds
X_test_s = (X_test - means) / stds

# Train logistic regression with simple gradient descent.
w = np.zeros(X_train_s.shape[1])
b = 0.0

for step in range(500):
    z = X_train_s @ w + b
    p = 1.0 / (1.0 + np.exp(-z))
    w -= 0.15 * (X_train_s.T @ (p - y_train)) / len(y_train)
    b -= 0.15 * np.mean(p - y_train)

# Predict with logistic regression probabilities.
logit_prob = 1.0 / (1.0 + np.exp(-(X_test_s @ w + b)))
logit_pred = (logit_prob >= 0.5).astype(int)

# Predict with manual three-nearest-neighbor distances.
knn_pred = []
for row in X_test_s:
    distances = np.sqrt(((X_train_s - row) ** 2).sum(axis=1))
    nearest = np.argsort(distances)[:3]
    knn_pred.append(int(y_train[nearest].mean() >= 0.5))

knn_pred = np.array(knn_pred)

# Predict with a simple decision tree rule.
tree_pred = ((X_test[:, 0] > 3.0) & (X_test[:, 1] > 16.0)).astype(int)

# Define compact classification metrics.
def metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    accuracy = (tp + tn) / len(y_true)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return accuracy, precision, recall, tp, fp, fn, tn

# Summarize models in a small table.
rows = []
for name, pred in [("Logistic", logit_pred), ("KNN", knn_pred), ("Tree rule", tree_pred)]:
    rows.append((name,) + metrics(y_test, pred))

summary = pd.DataFrame(rows, columns=["Model", "Accuracy", "Precision", "Recall", "TP", "FP", "FN", "TN"])

# Print concise results for comparison.
print("Manual classifier comparison on bridge risk test data")
print(summary.round(2).to_string(index=False))
print("Tree rule: high risk if crack > 3.0 mm and load > 16 tons")



### **3.2. Decision Tree Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_03_02.jpg?v=1783561631" width="250">



>* Test tree rules on unseen data
>* Use multiple metrics beyond accuracy

>* Precision shows trust in positive predictions
>* Recall finds positives based on error costs

>* Balance tree simplicity with predictive performance
>* Use metrics to judge interpretability tradeoffs



In [ ]:
#@title Python Code - Decision Tree Metrics

# Compare classifiers using small engineering data.
# Metrics reveal different decision tree weaknesses.
# Built-in calculations keep every step visible.

import numpy as np
import pandas as pd

# Create deterministic bridge inspection examples.
np.random.seed(7)
crack_mm = np.array([1, 2, 3, 4, 6, 7, 8, 10, 12, 14, 16, 18])
load_tons = np.array([5, 7, 6, 9, 10, 12, 11, 14, 15, 17, 18, 20])

# Label one means urgent repair needed.
y = np.array([0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1])
X = np.column_stack([crack_mm, load_tons])

# Use fixed train and test rows.
train_idx = np.array([0, 1, 2, 3, 4, 5, 7, 9])
test_idx = np.array([6, 8, 10, 11])

# Split features and labels safely.
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Confirm the tiny test set exists.
assert X_test.shape[0] == y_test.shape[0]
assert X_train.shape[1] == X_test.shape[1]

# Add intercept for logistic regression.
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0) + 1e-9

# Standardize features for stable gradients.
Z_train = (X_train - X_mean) / X_std
Z_test = (X_test - X_mean) / X_std

# Fit logistic regression manually.
w = np.zeros(Z_train.shape[1] + 1)
Zb = np.column_stack([np.ones(len(Z_train)), Z_train])

# Run a short silent gradient loop.
for step in range(400):
    scores = Zb @ w
    probs = 1 / (1 + np.exp(-scores))
    grad = Zb.T @ (probs - y_train) / len(y_train)
    w -= 0.25 * grad

# Predict logistic classes on test data.
Ztb = np.column_stack([np.ones(len(Z_test)), Z_test])
log_pred = ((1 / (1 + np.exp(-(Ztb @ w)))) >= 0.5).astype(int)

# Predict KNN using manual distances.
def knn_predict(row, k=3):
    distances = np.sqrt(((X_train - row) ** 2).sum(axis=1))
    nearest = np.argsort(distances)[:k]
    return int(y_train[nearest].mean() >= 0.5)

# Apply KNN to each test row.
knn_pred = np.array([knn_predict(row) for row in X_test])

# Use a simple interpretable tree rule.
tree_pred = ((X_test[:, 0] >= 10) | (X_test[:, 1] >= 16)).astype(int)

# Compute common classification metrics.
def metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    acc = (tp + tn) / len(y_true)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(tp + fn, 1)
    return acc, prec, rec

# Summarize models in a compact table.
rows = []
for name, pred in [("Logistic", log_pred), ("KNN", knn_pred), ("Tree rule", tree_pred)]:
    acc, prec, rec = metrics(y_test, pred)
    rows.append([name, round(acc, 2), round(prec, 2), round(rec, 2)])

# Display only concise teaching output.
results = pd.DataFrame(rows, columns=["Model", "Accuracy", "Precision", "Recall"])
print("Decision tree rule: urgent if crack >= 10 mm or load >= 16 tons")
print(results.to_string(index=False))



### **3.3. Interpreting Test Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_05/Lecture_B/image_03_03.jpg?v=1783561633" width="250">



>* Look beyond accuracy when comparing classifiers
>* Choose models with acceptable error tradeoffs

>* Link metrics to each classifier’s behavior
>* Balance accuracy with interpretability and deployment

>* Treat test metrics as future evidence
>* Balance performance gains with real-world consequences



In [ ]:
#@title Python Code - Interpreting Test Metrics

# Compare classifiers using practical test metrics.
# Small civil dataset keeps calculations transparent.
# Metrics reveal different engineering tradeoffs.

import math
import numpy as np
import pandas as pd

# Create compact inspection data.
data = pd.DataFrame({
    "crack_mm": [0.2, 0.4, 0.7, 1.1, 1.4, 1.8, 2.2, 2.7, 3.1, 3.6, 4.0, 4.5],
    "deflection_in": [0.03, 0.04, 0.05, 0.08, 0.10, 0.13, 0.16, 0.20, 0.24, 0.29, 0.33, 0.38],
    "needs_repair": [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
})

# Split data without shuffling.
train = data.iloc[:8].copy()
test = data.iloc[8:].copy()

# Validate the tiny split.
assert len(train) >= 4 and len(test) >= 2
features = ["crack_mm", "deflection_in"]

# Standardize using training statistics.
means = train[features].mean()
stds = train[features].std(ddof=0).replace(0, 1)

# Prepare scaled matrices.
X_train = ((train[features] - means) / stds).to_numpy()
y_train = train["needs_repair"].to_numpy()
X_test = ((test[features] - means) / stds).to_numpy()
y_test = test["needs_repair"].to_numpy()

# Fit logistic regression manually.
w = np.zeros(X_train.shape[1])
b = 0.0

# Use short gradient descent.
for step in range(600):
    z = X_train @ w + b
    p = 1 / (1 + np.exp(-z))
    w -= 0.15 * (X_train.T @ (p - y_train)) / len(y_train)
    b -= 0.15 * np.mean(p - y_train)

# Predict with logistic regression.
log_probs = 1 / (1 + np.exp(-(X_test @ w + b)))
log_pred = (log_probs >= 0.5).astype(int)

# Predict with manual KNN.
def knn_predict(row, k=3):
    distances = np.sqrt(((X_train - row) ** 2).sum(axis=1))
    nearest = np.argsort(distances)[:k]
    return int(y_train[nearest].mean() >= 0.5)

# Apply KNN to test rows.
knn_pred = np.array([knn_predict(row) for row in X_test])

# Predict with simple tree rule.
def tree_rule(row):
    crack = row["crack_mm"]
    deflection = row["deflection_in"]
    return int((crack >= 2.0) or (deflection >= 0.18))

# Apply interpretable rule.
tree_pred = test.apply(tree_rule, axis=1).to_numpy()

# Compute common classification metrics.
def metrics(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())

    accuracy = (tp + tn) / len(y_true)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return accuracy, precision, recall

# Summarize model comparisons.
models = {
    "Logistic regression": log_pred,
    "KNN, k=3": knn_pred,
    "Simple tree rule": tree_pred
}

# Print compact metric table.
print("Model comparison on held-out bridge inspections")
print("Model                 Accuracy Precision Recall")

# Show each model's tradeoff.
for name, pred in models.items():
    acc, prec, rec = metrics(y_test, pred)
    print(f"{name:<21} {acc:>6.2f}    {prec:>6.2f}  {rec:>5.2f}")

# Connect metrics to decisions.
print("Higher recall means fewer missed repair cases.")
print("Higher precision means fewer unnecessary repair alarms.")
print("Best model depends on inspection cost and safety risk.")



# <font color="#418FDE" size="6.5" uppercase>**KNN and Trees**</font>


In this lecture, you learned to:
- Implement k-nearest neighbors classification using manual distance calculations. 
- Explain how simple decision tree splits create interpretable engineering rules. 
- Compare logistic regression, k-nearest neighbors, and simple tree rules using test metrics. 

In the next Module (Module 6), we will go over 'None'